In [26]:
import pandas as pd
import os
import sys
from pathlib import Path
# Find the project root (the folder that contains src/config.py) and make src/ importable
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import MOLECULES_UNIQUE, MATRIX_IDS, TRAIN, TEST, MODEL_HANSEN, ensure_dir
from sklearn.model_selection import GroupShuffleSplit

In [27]:
input_path = MOLECULES_UNIQUE
df = pd.read_excel(input_path)
df.columns = df.columns.str.strip()

print(f"Fichier chargé : {input_path}")
print(f"Nombre de lignes : {len(df)}")
print(f"Colonnes : {df.columns.tolist()}")

Fichier chargé : ~/workspace/ProjectII/solubility_prediction_nmr/data/Fichier final (RT+sol)_with_smiles_Unique_CAS_Unique_CAS.xlsx
Nombre de lignes : 144
Colonnes : ['Column', 'Method', 'Sample Name', 'CAS', 'RT', 'Signal', 'Matrix ID Name', 'MeOH', 'ACN', 'DMSO', 'DCM', 'CHCl3', 'solute_smiles']


ici j'importe mon document avec toutes les molécule d'entrainement et de teste. on enlève les espaces parasites. ON print que le dossier est trouvé, combien de lignes le compose et le nom de toutes les colonnes

In [28]:
all_matrix_ids = sorted(df['Matrix ID Name'].dropna().unique().tolist())

print(f"Nombre de Matrix ID uniques : {len(all_matrix_ids)}")
print(f"Liste : {all_matrix_ids}")

import joblib
matrix_ids_path = MATRIX_IDS
joblib.dump(all_matrix_ids, matrix_ids_path)
print(f"Sauvegardé : {matrix_ids_path}")

Nombre de Matrix ID uniques : 1
Liste : ['matrix ID BlueBird 1']
Sauvegardé : /home/coralie/workspace/ProjectII/solubility_prediction_nmr/data/all_matrix_ids.joblib


In [29]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['solute_smiles']))

df_train = df.iloc[train_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

print(f"Train : {len(df_train)} lignes ({len(df_train)/len(df)*100:.1f}%)")
print(f"Test  : {len(df_test)} lignes ({len(df_test)/len(df)*100:.1f}%)")

Train : 122 lignes (84.7%)
Test  : 22 lignes (15.3%)


On fait une découpe dans la liste de tous les lignes, on en chosit 15% pour le teste et le n=42 s'assure que a chaque entrainement, le modèle découpe de la même manière. gss.split s'assure que toutes les molécules avec le même smiles aillent dans le même groupe. train et test font deux tableaux d'indices pour pouvoir dire quelle ligne va ou. ensuite on déléctionne les lignes qui correspondent aux indices qu'on a tiré au sort. Aussi on remet des indices qui commences à zéro. 

In [30]:
train_path = ensure_dir(TRAIN)
test_path = ensure_dir(TEST)

df_train.to_csv(train_path, index=False)
df_test.to_csv(test_path, index=False)

print(f"Test sauvegardé  : {test_path} ({len(df_test)} lignes)")

Test sauvegardé  : /home/coralie/workspace/ProjectII/solubility_prediction_nmr/data/test.csv (22 lignes)
